In [ ]:
print("Welcome to Document Change Detective!")

Welcome to Document Change Detective!


In [ ]:
old_document = """
Assignment deadline is Monday.
Students must submit their reports.
"""

new_document = """
Assignment deadline is Friday.
Students must submit their reports online.
"""

print(old_document)
print(new_document)


Assignment deadline is Monday.
Students must submit their reports.


Assignment deadline is Friday.
Students must submit their reports online.



In [ ]:
import re
from difflib import SequenceMatcher

def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text)

old_words = tokenize(old_document)
new_words = tokenize(new_document)

matcher = SequenceMatcher(
    None, old_words, new_words, autojunk=False
)

for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == "equal":
        continue

    old_part = old_words[i1:i2]
    new_part = new_words[j1:j2]

    if tag == "replace":
        print("Modified:", "".join(old_part),
              "->", "".join(new_part))
    elif tag == "delete":
        print("Deleted:", "".join(old_part))
    elif tag == "insert":
        print("Added:", "".join(new_part))

Modified: Monday -> Friday
Added: online


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [old_document, new_document]

vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(documents)

similarity = cosine_similarity(
    vectors[0], vectors[1]
)[0][0]

print(f"Document similarity: {similarity * 100:.2f}%")

Document similarity: 73.27%


In [3]:
!pip install python-docx

In [1]:
from google.colab import files

print("Upload the ORIGINAL Word document")
original_upload = files.upload()

print("Upload the UPDATED Word document")
updated_upload = files.upload()

Upload the ORIGINAL Word document


Saving Document_Original.docx to Document_Original.docx
Upload the UPDATED Word document


Saving Document_Updated.docx to Document_Updated.docx


In [4]:
from docx import Document

# Get the uploaded filenames
original_filename = list(original_upload.keys())[0]
updated_filename = list(updated_upload.keys())[0]

# Read both Word documents
original_doc = Document(original_filename)
updated_doc = Document(updated_filename)

# Extract text from each paragraph
original_paragraphs = [
    p.text for p in original_doc.paragraphs
]

updated_paragraphs = [
    p.text for p in updated_doc.paragraphs
]

print("ORIGINAL DOCUMENT")
for i, paragraph in enumerate(original_paragraphs, start=1):
    print(f"Paragraph {i}: {paragraph}")

print("\nUPDATED DOCUMENT")
for i, paragraph in enumerate(updated_paragraphs, start=1):
    print(f"Paragraph {i}: {paragraph}")

ORIGINAL DOCUMENT
Paragraph 1: Benefits of Cloud Computing
Paragraph 2: Original Version — Document Change Detective Test File
Paragraph 3: Introduction
Paragraph 4: Cloud computing allows users to access computing resources over the internet.
Paragraph 5: It reduces the need to buy and maintain expensive physical servers.
Paragraph 6: Main Benefits
Paragraph 7: Cloud services can scale up or down according to business needs.
Paragraph 8: Users can access files and applications from different locations.
Paragraph 9: Cloud computing can reduce infrastructure costs for small businesses.
Paragraph 10: Security
Paragraph 11: Providers offer tools such as access controls and data encryption.
Paragraph 12: Users should create strong passwords and enable multi-factor authentication.
Paragraph 13: Conclusion
Paragraph 14: Cloud computing offers flexibility, scalability, and convenient access to resources.

UPDATED DOCUMENT
Paragraph 1: Benefits of Cloud Computing
Paragraph 2: Updated Version —

In [5]:
import re
import html
from difflib import SequenceMatcher
from IPython.display import display, HTML

def compare_words(old, new):
    a = re.findall(r'\w+|[^\w\s]|\s+', old)
    b = re.findall(r'\w+|[^\w\s]|\s+', new)

    matcher = SequenceMatcher(None, a, b, autojunk=False)
    old_html, new_html = "", ""

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        x = html.escape("".join(a[i1:i2]))
        y = html.escape("".join(b[j1:j2]))

        if tag == "equal":
            old_html += x
            new_html += y
        elif tag == "delete":
            old_html += f'<mark style="background:#fecaca;color:#111">{x}</mark>'
        elif tag == "insert":
            new_html += f'<mark style="background:#bbf7d0;color:#111">{y}</mark>'
        else:
            old_html += f'<mark style="background:#fecaca;color:#111">{x}</mark>'
            new_html += f'<mark style="background:#bbf7d0;color:#111">{y}</mark>'

    return old_html, new_html


old = [p for p in original_paragraphs if p.strip()]
new = [p for p in updated_paragraphs if p.strip()]

matcher = SequenceMatcher(None, old, new, autojunk=False)
changes = []

for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == "equal":
        continue

    count = max(i2 - i1, j2 - j1)

    for k in range(count):
        old_text = old[i1 + k] if i1 + k < i2 else ""
        new_text = new[j1 + k] if j1 + k < j2 else ""

        old_html, new_html = compare_words(old_text, new_text)

        if not old_text:
            change_type = "Added paragraph"
        elif not new_text:
            change_type = "Deleted paragraph"
        else:
            change_type = "Modified paragraph"

        changes.append({
            "type": change_type,
            "old_number": i1 + k + 1 if old_text else "—",
            "new_number": j1 + k + 1 if new_text else "—",
            "old_html": old_html or "<i>Nothing removed</i>",
            "new_html": new_html or "<i>Nothing added</i>"
        })

if not changes:
    display(HTML("<h3>No changes found!</h3>"))
else:
    report = """
    <div style="background:white;color:#111;padding:18px">
    <h2>Document Change Detective</h2>
    <p>🔴 Removed text &nbsp; 🟢 Added text</p>
    """

    for i, c in enumerate(changes, start=1):
        report += f"""
        <div style="border:1px solid #ddd;padding:14px;
                    margin:18px 0 30px;border-radius:8px">
          <h3>Change {i}: {c['type']}</h3>
          <table style="width:100%;border-collapse:collapse;color:#111">
            <tr>
              <th style="text-align:left;padding:8px;background:#eee">
                Original — Paragraph {c['old_number']}
              </th>
              <th style="text-align:left;padding:8px;background:#eee">
                Updated — Paragraph {c['new_number']}
              </th>
            </tr>
            <tr>
              <td style="padding:12px;vertical-align:top;border:1px solid #ddd">
                {c['old_html']}
              </td>
              <td style="padding:12px;vertical-align:top;border:1px solid #ddd">
                {c['new_html']}
              </td>
            </tr>
          </table>
        </div>
        """

    report += "</div>"
    display(HTML(report))


Original — Paragraph 2,Updated — Paragraph 2
Original Version — Document Change Detective Test File,Updated Version — Document Change Detective Test File
Original — Paragraph 4,Updated — Paragraph 4
Cloud computing allows users to access computing resources over the internet.,Cloud computing allows users to access computing resources and services over the internet.
Original — Paragraph 5,Updated — Paragraph 5
It reduces the need to buy and maintain expensive physical servers.,It reduces the need to purchase and maintain expensive physical servers.
Original — Paragraph 7,Updated — Paragraph 7
Cloud services can scale up or down according to business needs.,Cloud services can scale up or down quickly according to changing business needs.
Original — Paragraph 8,Updated — Paragraph 8
Users can access files and applications from different locations.,Users can access files and applications from almost any location with an internet connection.
Original — Paragraph 9,Updated — Paragraph 9


In [9]:
import re
import html
from difflib import SequenceMatcher
from IPython.display import display, HTML

def tokenize(text):
    return re.findall(r'\w+|[^\w\s]|\s+', text)

def compare_text(old_text, new_text):
    old_tokens = tokenize(old_text)
    new_tokens = tokenize(new_text)

    matcher = SequenceMatcher(
        None, old_tokens, new_tokens, autojunk=False
    )

    old_html, new_html = "", ""
    added = deleted = modified = 0

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        old_part = html.escape("".join(old_tokens[i1:i2]))
        new_part = html.escape("".join(new_tokens[j1:j2]))

        if tag == "equal":
            old_html += old_part
            new_html += new_part

        elif tag == "delete":
            old_html += (
                f'<span style="background:#fecaca;color:#111">'
                f'{old_part}</span>'
            )
            deleted += 1

        elif tag == "insert":
            new_html += (
                f'<span style="background:#bbf7d0;color:#111">'
                f'{new_part}</span>'
            )
            added += 1

        elif tag == "replace":
            old_html += (
                f'<span style="background:#fecaca;color:#111">'
                f'{old_part}</span>'
            )
            new_html += (
                f'<span style="background:#bbf7d0;color:#111">'
                f'{new_part}</span>'
            )
            modified += 1

    return old_html, new_html, added, deleted, modified


old = [p for p in original_paragraphs if p.strip()]
new = [p for p in updated_paragraphs if p.strip()]

paragraph_matcher = SequenceMatcher(
    None, old, new, autojunk=False
)

changes = []

for tag, i1, i2, j1, j2 in paragraph_matcher.get_opcodes():
    if tag == "equal":
        continue

    count = max(i2 - i1, j2 - j1)

    for k in range(count):
        old_text = old[i1 + k] if i1 + k < i2 else ""
        new_text = new[j1 + k] if j1 + k < j2 else ""

        if old_text == new_text:
            continue

        old_html, new_html, added, deleted, modified = compare_text(
            old_text, new_text
        )

        if not old_text:
            kind = "Added paragraph"
        elif not new_text:
            kind = "Deleted paragraph"
        else:
            kind = "Modified paragraph"

        changes.append({
            "type": kind,
            "old_no": i1 + k + 1 if old_text else "—",
            "new_no": j1 + k + 1 if new_text else "—",
            "old_html": old_html or "<i>No original text</i>",
            "new_html": new_html or "<i>No updated text</i>",
            "added": added,
            "deleted": deleted,
            "modified": modified
        })

print("Comparison ready!")
print("Changed sections:", len(changes))
print("Added text sections:", sum(c["added"] for c in changes))
print("Deleted text sections:", sum(c["deleted"] for c in changes))
print("Modified text sections:", sum(c["modified"] for c in changes))

Comparison ready!
Changed sections: 9
Added text sections: 7
Deleted text sections: 0
Modified text sections: 5


In [13]:
show_changes("All changes")

Original - Paragraph 2,Updated - Paragraph 2
Original Version — Document Change Detective Test File,Updated Version — Document Change Detective Test File
Original - Paragraph 4,Updated - Paragraph 4
Cloud computing allows users to access computing resources over the internet.,Cloud computing allows users to access computing resources and services over the internet.
Original - Paragraph 5,Updated - Paragraph 5
It reduces the need to buy and maintain expensive physical servers.,It reduces the need to purchase and maintain expensive physical servers.
Original - Paragraph 7,Updated - Paragraph 7
Cloud services can scale up or down according to business needs.,Cloud services can scale up or down quickly according to changing business needs.
Original - Paragraph 8,Updated - Paragraph 8
Users can access files and applications from different locations.,Users can access files and applications from almost any location with an internet connection.
Original - Paragraph 9,Updated - Paragraph 9


In [12]:
from IPython.display import display, HTML

def show_changes(filter_type="All changes"):
    selected = changes

    if filter_type == "Added paragraphs":
        selected = [
            c for c in changes
            if c["type"] == "Added paragraph"
        ]
    elif filter_type == "Deleted paragraphs":
        selected = [
            c for c in changes
            if c["type"] == "Deleted paragraph"
        ]
    elif filter_type == "Modified paragraphs":
        selected = [
            c for c in changes
            if c["type"] == "Modified paragraph"
        ]

    output = """
    <div style="font-family:Arial;color:#222">
    <h2>Document Change Detective</h2>
    <p>🔴 Removed text &nbsp; 🟢 Added text</p>
    """

    if not selected:
        output += "<p>No changes found for this filter.</p>"

    for number, c in enumerate(selected, 1):
        output += f"""
        <div style="border:1px solid #ddd;border-radius:8px;
                    padding:14px;margin:14px 0">
          <h3>Change {number}: {c['type']}</h3>
          <table style="width:100%;border-collapse:collapse">
            <tr>
              <th style="text-align:left;padding:8px;background:#eee">
                Original - Paragraph {c['old_no']}
              </th>
              <th style="text-align:left;padding:8px;background:#eee">
                Updated - Paragraph {c['new_no']}
              </th>
            </tr>
            <tr>
              <td style="border:1px solid #ddd;padding:12px">
                {c['old_html']}
              </td>
              <td style="border:1px solid #ddd;padding:12px">
                {c['new_html']}
              </td>
            </tr>
          </table>
        </div>
        """

    output += "</div>"
    display(HTML(output))

In [14]:
from docx import Document
from IPython.display import display, HTML

# Load both Word documents
original_doc = Document(original_filename)
updated_doc = Document(updated_filename)

# Extract all non-empty paragraphs
original_paragraphs = [
    p.text for p in original_doc.paragraphs if p.text.strip()
]
updated_paragraphs = [
    p.text for p in updated_doc.paragraphs if p.text.strip()
]

print("Both documents loaded successfully!")
print("Original paragraphs:", len(original_paragraphs))
print("Updated paragraphs:", len(updated_paragraphs))

Both documents loaded successfully!
Original paragraphs: 14
Updated paragraphs: 15


In [15]:

from difflib import SequenceMatcher
from IPython.display import display, HTML
from html import escape
import re

def highlight_diff(old_text, new_text):
    old_words = re.findall(r'\s+|[^\s]+', old_text)
    new_words = re.findall(r'\s+|[^\s]+', new_text)

    matcher = SequenceMatcher(None, old_words, new_words)
    old_html, new_html = "", ""

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        old_part = escape("".join(old_words[i1:i2]))
        new_part = escape("".join(new_words[j1:j2]))

        if tag == "equal":
            old_html += old_part
            new_html += new_part
        elif tag == "delete":
            old_html += f'<span class="removed">{old_part}</span>'
        elif tag == "insert":
            new_html += f'<span class="added">{new_part}</span>'
        elif tag == "replace":
            old_html += f'<span class="removed">{old_part}</span>'
            new_html += f'<span class="added">{new_part}</span>'

    return old_html, new_html


matcher = SequenceMatcher(
    None, original_paragraphs, updated_paragraphs
)

rows = []
old_num = new_num = 0

for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == "equal":
        for k in range(i2 - i1):
            old_num += 1
            new_num += 1
            old_html, new_html = highlight_diff(
                original_paragraphs[i1 + k],
                updated_paragraphs[j1 + k]
            )
            rows.append((old_num, new_num, old_html, new_html))

    elif tag == "replace":
        old_block = original_paragraphs[i1:i2]
        new_block = updated_paragraphs[j1:j2]
        count = max(len(old_block), len(new_block))

        for k in range(count):
            old_num += 1 if k < len(old_block) else 0
            new_num += 1 if k < len(new_block) else 0
            old_text = old_block[k] if k < len(old_block) else ""
            new_text = new_block[k] if k < len(new_block) else ""
            old_html, new_html = highlight_diff(old_text, new_text)
            rows.append((old_num if old_text else "—",
                         new_num if new_text else "—",
                         old_html or "—", new_html or "—"))

    elif tag == "delete":
        for paragraph in original_paragraphs[i1:i2]:
            old_num += 1
            rows.append((old_num, "—", escape(paragraph), "—"))

    elif tag == "insert":
        for paragraph in updated_paragraphs[j1:j2]:
            new_num += 1
            rows.append(("—", new_num, "—", escape(paragraph)))

table_rows = ""
for old_n, new_n, old_html, new_html in rows:
    table_rows += f"""
    <tr>
      <td class="num">{old_n}</td><td>{old_html}</td>
      <td class="num">{new_n}</td><td>{new_html}</td>
    </tr>
    """

display(HTML("""
<style>
.compare {width:100%; border-collapse:collapse; color:#222;}
.compare th {background:#e9edf3; padding:10px; text-align:left;}
.compare td {border:1px solid #ddd; padding:10px;
             vertical-align:top; white-space:pre-wrap; overflow-wrap:anywhere;}
.compare .num {width:45px; color:#555; background:#f7f7f7;}
.removed {background:#ffd6d6; color:#9b1c1c; text-decoration:line-through;}
.added {background:#d6f5df; color:#176b36;}
</style>
<h2>Document Change Detective — Side-by-Side Comparison</h2>
<p><span style="background:#ffd6d6">Red = removed text</span> &nbsp;
<span style="background:#d6f5df">Green = added text</span></p>
<div style="overflow-x:auto">
<table class="compare">
<thead><tr>
<th colspan="2">Original Document</th>
<th colspan="2">Updated Document</th>
</tr><tr>
<th>Paragraph</th><th>Content</th>
<th>Paragraph</th><th>Content</th>
</tr></thead>
<tbody>
""" + table_rows + """
</tbody></table></div>
"""))


In [20]:

from difflib import SequenceMatcher
from IPython.display import display, HTML
from docx import Document
from html import escape
import re

# 1. Load both Word documents
old_doc = Document(original_filename)
new_doc = Document(updated_filename)

old_paras = [p.text for p in old_doc.paragraphs if p.text.strip()]
new_paras = [p.text for p in new_doc.paragraphs if p.text.strip()]


# 2. Compare words and highlight differences
def tokenize(text):
    return re.findall(r"\w+(?:['’-]\w+)*|[^\w\s]|\s+", text)


def compare_words(old_text, new_text):
    old_tokens = tokenize(old_text)
    new_tokens = tokenize(new_text)

    matcher = SequenceMatcher(
        None, old_tokens, new_tokens, autojunk=False
    )

    old_html, new_html = "", ""

    for tag, a, b, c, d in matcher.get_opcodes():
        old_part = escape("".join(old_tokens[a:b]))
        new_part = escape("".join(new_tokens[c:d]))

        if tag == "equal":
            old_html += old_part
            new_html += new_part
        elif tag == "delete":
            old_html += f'<span class="deleted">{old_part}</span>'
        elif tag == "insert":
            new_html += f'<span class="added">{new_part}</span>'
        else:
            old_html += f'<span class="deleted">{old_part}</span>'
            new_html += f'<span class="added">{new_part}</span>'

    return old_html, new_html


# 3. Align paragraphs, including inserted/deleted paragraphs
def similarity(a, b):
    return SequenceMatcher(
        None, a.casefold(), b.casefold(), autojunk=False
    ).ratio()


def align_paragraphs(old_list, new_list):
    n, m = len(old_list), len(new_list)
    gap = 0.55

    dp = [[0.0] * (m + 1) for _ in range(n + 1)]
    back = [[None] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        dp[i][0] = dp[i - 1][0] - gap
        back[i][0] = "delete"

    for j in range(1, m + 1):
        dp[0][j] = dp[0][j - 1] - gap
        back[0][j] = "insert"

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sim = similarity(old_list[i - 1], new_list[j - 1])

            choices = [
                (dp[i - 1][j - 1] + (2 * sim - 1), "match"),
                (dp[i - 1][j] - gap, "delete"),
                (dp[i][j - 1] - gap, "insert")
            ]

            dp[i][j], back[i][j] = max(
                choices, key=lambda item: item[0]
            )

    aligned = []
    i, j = n, m

    while i > 0 or j > 0:
        move = back[i][j]

        if move == "match":
            aligned.append((old_list[i - 1], new_list[j - 1]))
            i -= 1
            j -= 1
        elif move == "delete":
            aligned.append((old_list[i - 1], None))
            i -= 1
        else:
            aligned.append((None, new_list[j - 1]))
            j -= 1

    return aligned[::-1]


aligned = align_paragraphs(old_paras, new_paras)

left_rows, right_rows = [], []
old_number = new_number = 0
added_count = deleted_count = modified_count = 0

for old_text, new_text in aligned:
    if old_text is not None:
        old_number += 1
    if new_text is not None:
        new_number += 1

    if old_text is not None and new_text is not None:
        if old_text == new_text:
            old_html = new_html = escape(old_text)
        else:
            old_html, new_html = compare_words(old_text, new_text)
            modified_count += 1

        left_rows.append((old_number, old_html))
        right_rows.append((new_number, new_html))

    elif old_text is not None:
        deleted_count += 1
        left_rows.append((
            old_number,
            f'<span class="deleted">{escape(old_text)}</span>'
        ))
        right_rows.append((
            "—", '<em class="missing">Paragraph deleted</em>'
        ))

    else:
        added_count += 1
        left_rows.append((
            "—", '<em class="missing">New paragraph added</em>'
        ))
        right_rows.append((
            new_number,
            f'<span class="added">{escape(new_text)}</span>'
        ))


# 4. Calculate similarity
old_all = "\n".join(old_paras)
new_all = "\n".join(new_paras)

overall_similarity = SequenceMatcher(
    None, old_all, new_all, autojunk=False
).ratio() * 100


# 5. Create the two document panels
def make_panel(title, rows):
    content = ""

    for number, text in rows:
        content += (
            f'<div class="paragraph">'
            f'<span class="number">{number}</span>'
            f'<span class="text">{text}</span>'
            f'</div>'
        )

    return f'<section><h3>{title}</h3>{content}</section>'


# 6. Display the final comparison
page = f"""
<style>
.comparison {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 14px;
    font-family: Arial, sans-serif;
    color: #222222;
}}

.comparison section {{
    background: #ffffff;
    border: 1px solid #cccccc;
    border-radius: 8px;
    overflow: hidden;
    min-width: 0;
}}

.comparison h3 {{
    background: #e9edf3;
    color: #222222;
    margin: 0;
    padding: 14px;
    border-bottom: 1px solid #cccccc;
    font-size: 16px;
    font-weight: bold;
}}

.paragraph {{
    display: flex;
    gap: 10px;
    padding: 12px;
    border-bottom: 1px solid #eeeeee;
    line-height: 1.6;
    overflow-wrap: anywhere;
}}

.number {{
    color: #666666;
    min-width: 25px;
    font-size: 12px;
}}

.text {{
    white-space: pre-wrap;
}}

.deleted {{
    background: #ffdada;
    color: #8b1717;
    text-decoration: line-through;
}}

.added {{
    background: #d5f5df;
    color: #145c2e;
}}

.missing {{
    color: #777777;
    font-size: 13px;
}}

.summary {{
    display: flex;
    flex-wrap: wrap;
    gap: 12px;
    margin: 14px 0;
    font-family: Arial, sans-serif;
}}

.metric {{
    padding: 14px;
    border-radius: 8px;
    min-width: 130px;
}}

@media (max-width: 650px) {{
    .comparison {{ grid-template-columns: 1fr; }}
}}
</style>

<h2 style="color:#222222; background:#f3f4f6;
           padding:14px; border-radius:8px;">
    Document Change Detective
</h2>

<div class="summary">
    <div class="metric" style="background:#e0e7ff;color:#1e3a8a;">
        <b>Similarity</b><br>
        <span style="font-size:24px;">{overall_similarity:.1f}%</span>
    </div>
    <div class="metric" style="background:#fef3c7;color:#92400e;">
        <b>Modified paragraphs</b><br>
        <span style="font-size:24px;">{modified_count}</span>
    </div>
    <div class="metric" style="background:#d5f5df;color:#145c2e;">
        <b>Added paragraphs</b><br>
        <span style="font-size:24px;">{added_count}</span>
    </div>
    <div class="metric" style="background:#ffdada;color:#8b1717;">
        <b>Deleted paragraphs</b><br>
        <span style="font-size:24px;">{deleted_count}</span>
    </div>
</div>

<p>
    <span style="background:#ffdada;color:#8b1717;">
        Red = removed text
    </span>
    &nbsp;&nbsp;
    <span style="background:#d5f5df;color:#145c2e;">
        Green = added text
    </span>
</p>

<div class="comparison">
    {make_panel("ORIGINAL DOCUMENT", left_rows)}
    {make_panel("UPDATED DOCUMENT", right_rows)}
</div>
"""

display(HTML(page))


In [17]:

from difflib import SequenceMatcher

old_text = "\n".join(original_paragraphs)
new_text = "\n".join(updated_paragraphs)

matcher = SequenceMatcher(None, old_text, new_text)
added_chars = 0
deleted_chars = 0

for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == "insert":
        added_chars += j2 - j1
    elif tag == "delete":
        deleted_chars += i2 - i1
    elif tag == "replace":
        deleted_chars += i2 - i1
        added_chars += j2 - j1

similarity = matcher.ratio() * 100

print("📊 DOCUMENT CHANGE SUMMARY")
print("-" * 32)
print(f"Similarity: {similarity:.1f}%")
print(f"Characters added: {added_chars}")
print(f"Characters deleted: {deleted_chars}")
print(f"Original paragraphs: {len(original_paragraphs)}")
print(f"Updated paragraphs: {len(updated_paragraphs)}")


📊 DOCUMENT CHANGE SUMMARY
--------------------------------
Similarity: 46.2%
Characters added: 543
Characters deleted: 330
Original paragraphs: 14
Updated paragraphs: 15


In [21]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 29.6 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 115.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pymupdf]


In [22]:
import fitz  # PyMuPDF
from google.colab import files

def extract_pdf_text(pdf_file):
    document = fitz.open(pdf_file)
    pages = []

    for page_number, page in enumerate(document, start=1):
        text = page.get_text("text")
        pages.append({
            "page": page_number,
            "text": text
        })

    document.close()
    return pages

print("PDF text extraction is ready!")

PDF text extraction is ready!


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [23]:

from google.colab import files
import fitz

print("Upload the ORIGINAL PDF")
original_pdf_upload = files.upload()

print("Upload the UPDATED PDF")
updated_pdf_upload = files.upload()

original_pdf_filename = list(original_pdf_upload.keys())[0]
updated_pdf_filename = list(updated_pdf_upload.keys())[0]

original_pdf_pages = extract_pdf_text(original_pdf_filename)
updated_pdf_pages = extract_pdf_text(updated_pdf_filename)

original_pdf_text = "\n".join(
    page["text"] for page in original_pdf_pages
)
updated_pdf_text = "\n".join(
    page["text"] for page in updated_pdf_pages
)

print("\nBoth PDFs loaded successfully!")
print("Original PDF pages:", len(original_pdf_pages))
print("Updated PDF pages:", len(updated_pdf_pages))
print("Original PDF characters:", len(original_pdf_text))
print("Updated PDF characters:", len(updated_pdf_text))


Upload the ORIGINAL PDF


Saving College_Trip_Original.pdf to College_Trip_Original.pdf
Upload the UPDATED PDF


Saving College_Trip_Updated.pdf to College_Trip_Updated.pdf

Both PDFs loaded successfully!
Original PDF pages: 1
Updated PDF pages: 2
Original PDF characters: 1422
Updated PDF characters: 1743


In [26]:

from IPython.display import display, HTML

page = f"""
<style>
.pdf-comparison {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 14px;
    font-family: Arial, sans-serif;
    color: #222222;
}}
.pdf-comparison section {{
    background: #ffffff;
    color: #222222;
    border: 1px solid #cccccc;
    border-radius: 8px;
    overflow: hidden;
    min-width: 0;
}}
.pdf-comparison h3 {{
    background: #e9edf3;
    color: #222222;
    margin: 0;
    padding: 14px;
    font-size: 16px;
}}
.paragraph {{
    display: flex;
    gap: 10px;
    padding: 10px;
    border-bottom: 1px solid #eeeeee;
    line-height: 1.6;
    overflow-wrap: anywhere;
    background: #ffffff;
    color: #222222;
}}
.number {{
    color: #555555;
    min-width: 25px;
    font-size: 12px;
}}
.text {{ white-space: pre-wrap; color: #222222; }}
.deleted {{
    background: #ffcaca;
    color: #8b1717;
    text-decoration: line-through;
}}
.added {{ background: #c6f6d5; color: #145c2e; }}
@media (max-width: 650px) {{
    .pdf-comparison {{ grid-template-columns: 1fr; }}
}}
</style>

<h2 style="color:#ffffff;">PDF Document Change Detective</h2>
<p style="color:#ffffff;"><b>Text similarity:</b> {similarity:.1f}%</p>
<p>
<span style="background:#ffcaca;color:#8b1717;padding:3px;">
Red = deleted text</span>
&nbsp;
<span style="background:#c6f6d5;color:#145c2e;padding:3px;">
Green = added text</span>
</p>

<div class="pdf-comparison">
{make_panel("ORIGINAL PDF", left_rows)}
{make_panel("UPDATED PDF", right_rows)}
</div>
"""

display(HTML(page))


In [27]:

from docx import Document

doc = Document(original_filename)

print("WORD DOCUMENT FORMATTING REPORT")
print("-" * 35)

print("\nParagraphs:", len(doc.paragraphs))
print("Tables:", len(doc.tables))
print("Sections:", len(doc.sections))

print("\nHeadings found:")
for i, paragraph in enumerate(doc.paragraphs, start=1):
    if paragraph.style.name.startswith("Heading"):
        print(f"Paragraph {i}: {paragraph.style.name} - {paragraph.text}")

print("\nTable details:")
for i, table in enumerate(doc.tables, start=1):
    print(f"Table {i}: {len(table.rows)} rows, {len(table.columns)} columns")

print("\nFormatting inspection complete!")


WORD DOCUMENT FORMATTING REPORT
-----------------------------------

Paragraphs: 14
Tables: 0
Sections: 1

Headings found:
Paragraph 3: Heading 1 - Introduction
Paragraph 6: Heading 1 - Main Benefits
Paragraph 10: Heading 1 - Security
Paragraph 13: Heading 1 - Conclusion

Table details:

Formatting inspection complete!


In [28]:

from docx import Document

doc = Document(original_filename)

print("WORD TEXT FORMATTING REPORT")
print("-" * 35)

for p_num, paragraph in enumerate(doc.paragraphs, start=1):
    if not paragraph.text.strip():
        continue

    print(f"\nParagraph {p_num}")
    print("Style:", paragraph.style.name)
    print("Text:", paragraph.text)

    for run in paragraph.runs:
        if run.text.strip():
            formats = []
            if run.bold:
                formats.append("Bold")
            if run.italic:
                formats.append("Italic")
            if run.underline:
                formats.append("Underline")

            if formats:
                print(f"  {run.text!r} → {', '.join(formats)}")

print("\nFormatting inspection complete!")


WORD TEXT FORMATTING REPORT
-----------------------------------

Paragraph 1
Style: Normal
Text: Benefits of Cloud Computing
  'Benefits of Cloud Computing' → Bold

Paragraph 2
Style: Normal
Text: Original Version — Document Change Detective Test File
  'Original Version — Document Change Detective Test File' → Italic

Paragraph 3
Style: Heading 1
Text: Introduction

Paragraph 4
Style: Normal
Text: Cloud computing allows users to access computing resources over the internet.

Paragraph 5
Style: Normal
Text: It reduces the need to buy and maintain expensive physical servers.

Paragraph 6
Style: Heading 1
Text: Main Benefits

Paragraph 7
Style: Normal
Text: Cloud services can scale up or down according to business needs.

Paragraph 8
Style: Normal
Text: Users can access files and applications from different locations.

Paragraph 9
Style: Normal
Text: Cloud computing can reduce infrastructure costs for small businesses.

Paragraph 10
Style: Heading 1
Text: Security

Paragraph 11
Style: No

In [29]:
from docx import Document

doc = Document(updated_filename)

print("UPDATED DOCUMENT FORMATTING REPORT")
print("-" * 35)

for i, p in enumerate(doc.paragraphs, start=1):
    if not p.text.strip():
        continue

    print(f"\nParagraph {i} | Style: {p.style.name}")
    print(p.text)

    for run in p.runs:
        if run.text.strip():
            formats = []
            if run.bold:
                formats.append("Bold")
            if run.italic:
                formats.append("Italic")
            if run.underline:
                formats.append("Underline")

            if formats:
                print(f"  {run.text!r} → {', '.join(formats)}")

print("\nInspection complete!")

UPDATED DOCUMENT FORMATTING REPORT
-----------------------------------

Paragraph 1 | Style: Normal
Benefits of Cloud Computing
  'Benefits of Cloud Computing' → Bold

Paragraph 2 | Style: Normal
Updated Version — Document Change Detective Test File
  'Updated Version — Document Change Detective Test File' → Italic

Paragraph 3 | Style: Heading 1
Introduction

Paragraph 4 | Style: Normal
Cloud computing allows users to access computing resources and services over the internet.

Paragraph 5 | Style: Normal
It reduces the need to purchase and maintain expensive physical servers.

Paragraph 6 | Style: Heading 1
Main Benefits

Paragraph 7 | Style: Normal
Cloud services can scale up or down quickly according to changing business needs.

Paragraph 8 | Style: Normal
Users can access files and applications from almost any location with an internet connection.

Paragraph 9 | Style: Normal
Cloud computing can reduce infrastructure costs for small and medium-sized businesses.

Paragraph 10 | Styl